# Day 3 — LLM APIs & Chatbots

In this module, we interface with Large Language Models programmatically. We make raw HTTP requests using the Python `requests` library to interact with local (Ollama) and cloud (Groq) models, handle streaming responses, and manage conversational memory.

## Objectives
- Query LLMs directly using POST requests.
- Implement word-by-word real-time output streaming.
- Maintain multi-turn conversational context.
- **Extended Implementation:** Build a persistent chat logger (JSON), construct a token usage estimator, and implement a dynamic system prompt persona switcher.

In [ ]:
# Install libraries
!pip install requests python-dotenv -q
print("✅ Libraries installed.")

## The API Concept: JSON Request and Response
We query models by passing JSON payloads specifying the model parameters and input message dictionary, then decode the JSON response containing the text completion.

In [ ]:
import os
from dotenv import load_dotenv
import requests

load_dotenv() # Load API keys from .env file
groq_api_key = os.getenv("GROQ_API_KEY")

# Check if local Ollama or Groq is available
print("Groq API Key available:", groq_api_key is not None and len(groq_api_key) > 0)

## 1. Requesting Model Completions
We query the Groq API using Python `requests`. The request format is identical whether accessing cloud providers or local setups (e.g. Ollama).

In [ ]:
def query_llm(prompt, system_instruction=None, model="llama-3.1-8b-instant"):
    if not groq_api_key:
        raise ValueError("Please set your GROQ_API_KEY in the environment or .env file.")
        
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {groq_api_key}",
        "Content-Type": "application/json"
    }
    
    messages = []
    if system_instruction:
        messages.append({"role": "system", "content": system_instruction})
    messages.append({"role": "user", "content": prompt})
    
    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.7
    }
    
    response = requests.post(url, json=payload, headers=headers)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Error: {response.status_code} - {response.text}"

answer = query_llm("Explain what an API is in 2 sentences.", "You are talking to a 10 year old.")
print("Response:\n", answer)

## 2. Streaming Responses
We request streamed outputs where tokens are yielded as they are generated by the model.

In [ ]:
import json

def stream_llm(prompt, model="llama-3.1-8b-instant"):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {groq_api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    response = requests.post(url, json=payload, headers=headers, stream=True)
    
    for line in response.iter_lines():
        if line:
            line_str = line.decode('utf-8')
            if line_str.startswith("data: "):
                content = line_str[6:]
                if content.strip() == "[DONE]":
                    break
                try:
                    chunk = json.loads(content)
                    delta = chunk['choices'][0]['delta'].get('content', '')
                    print(delta, end="", flush=True)
                except Exception:
                    pass
    print() 

stream_llm("Write a short haiku about coding.")

## 3. Multi-Turn Conversations and Memory
We maintain context in multi-turn dialogues by storing conversational history and appending message dictionaries to our payloads.

In [ ]:
class SimpleChatbot:
    def __init__(self, system_prompt="You are a helpful assistant.", model="llama-3.1-8b-instant"):
        self.model = model
        self.history = [{"role": "system", "content": system_prompt}]
        self.url = "https://api.groq.com/openai/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {groq_api_key}",
            "Content-Type": "application/json"
        }
        
    def chat(self, user_msg):
        self.history.append({"role": "user", "content": user_msg})
        
        payload = {
            "model": self.model,
            "messages": self.history,
            "temperature": 0.7
        }
        
        response = requests.post(self.url, json=payload, headers=self.headers)
        if response.status_code == 200:
            reply = response.json()['choices'][0]['message']['content']
            self.history.append({"role": "assistant", "content": reply})
            return reply
        else:
            return f"Error: {response.text}"

bot = SimpleChatbot(system_prompt="You are a grumpy old wizard. Speak in riddles.")
print("Turn 1:", bot.chat("Hi, I am lost. Can you help me?"))
print("Turn 2:", bot.chat("What is my problem again? Can you remind me?"))

---
# Extended Implementation
Below are implementation utilities for persistence, token tracking, and switching user personas.

### 1. Persistent Chat Logger (JSON)
We implement methods to save and reload conversation history files.

In [ ]:
def save_chat_history(history, filepath="chat_log.json"):
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=2)
    print(f"💾 Chat history saved to {filepath}")

def load_chat_history(filepath="chat_log.json"):
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            history = json.load(f)
        print(f"Loaded {len(history)} messages from {filepath}")
        return history
    return []

# Save Wiz chatbot's history
save_chat_history(bot.history)

# Reload history and print message counts
reloaded = load_chat_history()
for msg in reloaded:
    print(f"[{msg['role'].upper()}]: {msg['content'][:60]}...")

### 2. Token & Cost Estimator
We estimate token consumption and display mock pricing metrics based on average character lengths.

In [ ]:
def estimate_usage(prompt, response_text, model="llama-3.1-8b-instant"):
    # Estimate tokens (characters / 4 is a common heuristic)
    input_tokens = max(1, len(prompt) // 4)
    output_tokens = max(1, len(response_text) // 4)
    total_tokens = input_tokens + output_tokens
    
    # Approximate costs ($0.05 input per 1M, $0.08 output per 1M)
    input_cost = (input_tokens / 1_000_000) * 0.05
    output_cost = (output_tokens / 1_000_000) * 0.08
    total_cost = input_cost + output_cost
    
    return {
        "Input Tokens (Estimated)": input_tokens,
        "Output Tokens (Estimated)": output_tokens,
        "Total Cost (USD)": f"${total_cost:.8f}"
    }

usage_stats = estimate_usage("Explain what an API is in 2 sentences.", answer)
print("Estimated Session Cost:")
for k, v in usage_stats.items():
    print(f"  {k}: {v}")

### 3. Dynamic System Prompt Persona Switcher
We implement a custom chatbot class that supports changing the system personality prompt during execution.

In [ ]:
class PersonaBot(SimpleChatbot):
    def switch_persona(self, new_persona_prompt):
        if self.history and self.history[0]['role'] == 'system':
            print(f"Switching persona to: '{new_persona_prompt}'")
            self.history[0]['content'] = new_persona_prompt
        else:
            self.history.insert(0, {"role": "system", "content": new_persona_prompt})

p_bot = PersonaBot(system_prompt="You are a helpful Python code reviewer.")
print("Review response:", p_bot.chat("def add(a,b): return a+b"))

# Switch persona mid-way
p_bot.switch_persona("You are a Shakespearean actor. Reply in Elizabethan English.")
print("Shakespearean response:", p_bot.chat("Can you write a basic function for multiplication?"))